In [9]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.globals import set_debug
from typing import Literal

from pydantic import BaseModel, Field
from datasets import load_dataset

In [15]:
# Define prompt template
classification_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an experienced newspaper editor who has experience working with content across news categories. "
        "Classify the article into exactly one of: World, Sports, Business, Sci/Tech. "
        "Respond with the category and a brief one-sentence reason."
    )),
    ("human", "{text}"),
])


In [16]:
# Define response schema
class ClassificationResult(BaseModel):
    category: Literal["World", "Sports", "Business", "Sci/Tech"] = Field(
        description="The news category that best describes the content. Valid values are World, Sports, Business, Sci/Tech"
    )
    reasoning: str = Field(
        description="A brief one-sentence explanation of why this category was chosen"
    )


In [17]:
# Define model
model_name = "qwen2.5:3b"

llm = ChatOllama(model=model_name)
structured_llm = llm.with_structured_output(ClassificationResult)

In [18]:
# Create chain (RunnableSequence)
chain = classification_prompt | structured_llm

### Load data

In [6]:
def load_sample(split: str = "test", n: int = 100):
    ds = load_dataset("fancyzhx/ag_news")
    return ds[split].select(range(n))

sample_data = load_sample("test", 10)

### Generate response
- with debug

In [19]:
set_debug(True)
response = chain.invoke({"text": sample_data[0]["text"]})
set_debug(False)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "text": "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul."
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "text": "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul."
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "System: You are an experienced newspaper editor who has experience working with content across news categories. Classify the article into exactly one of: World, Sports, Business, Sci/Tech. Respond with the category and a brief one-sentence reason.\nHuman: Fear